# 7D12 VHH 结构预测 - Google Colab

本 notebook 使用多种方法预测 7D12 VHH 的三维结构：
- AlphaFold2 (ColabFold)
- ESMFold
- IgFold (抗体特异性)

## 原始序列
```
QVQLVESGGGLVQVGGSLRLSRALSGFWYNHMGWFRQAPGKEREGVAVITADSGSTTYADSVKGRFTISRDDARNTVYLQMNSLKPEDTAVYYCAAGGVGWPYFDYWGQGTQVTVSS
```
- **长度**: 117 aa
- **靶点**: EGFR
- **来源**: 羊驼 (Alpaca)

## 方法 1: ColabFold (AlphaFold2) - 推荐 ⭐

In [ ]:
# 安装 ColabFold
!pip install -q "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold.git"
!pip install -q py3Dmol

In [ ]:
from colabfold.batch import batch_predict
from colabfold.utils import setup_environment
import py3Dmol
import pandas as pd

# 设置环境
setup_environment(use_templates=False, use_amber=False)

# 7D12 VHH 序列
sequence = "QVQLVESGGGLVQVGGSLRLSRALSGFWYNHMGWFRQAPGKEREGVAVITADSGSTTYADSVKGRFTISRDDARNTVYLQMNSLKPEDTAVYYCAAGGVGWPYFDYWGQGTQVTVSS"

print(f"序列长度: {len(sequence)} aa")
print(f"序列: {sequence}")

In [ ]:
# 运行 AlphaFold2 预测
result_dir = "7d12_alphafold_results"

# 准备输入
queries = [
    ("7D12_VHH", sequence)
]

# 运行预测（可能需要 10-30 分钟）
batch_predict(
    queries,
    result_dir=result_dir,
    use_templates=False,
    use_amber=False,
    num_recycles=3,
    num_models=5
)

print("\n预测完成！")

In [ ]:
# 可视化最佳模型
import os
from pathlib import Path

# 查找生成的 PDB 文件
result_path = Path(result_dir)
pdb_files = list(result_path.glob("*.pdb"))

if pdb_files:
    # 选择排名第一的模型（通常是最佳模型）
    best_model = sorted(pdb_files)[0]
    print(f"使用模型: {best_model.name}")
    
    # 读取 PDB 文件
    with open(best_model, 'r') as f:
        pdb_str = f.read()
    
    # 3D 可视化
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_str, 'pdb')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.addStyle({'and': [{'resn': ['GFWYNH']}, {'chain': 'A'}]}, 
                 {'stick': {'colorscheme': 'greenCarbon'}})
    view.zoomTo()
    view.show()
    
    # 显示置信度分数
    print("\n模型文件已生成，可以下载查看详细信息")
else:
    print("未找到 PDB 文件")

## 方法 2: ESMFold (快速预测)

In [ ]:
# 安装 ESMFold
!pip install -q fair-esm
!pip install -q py3Dmol

In [ ]:
import torch
import esm
import py3Dmol

# 加载 ESMFold 模型
model = esm.pretrained.esmfold_v1()
model = model.eval().cuda()  # 使用 GPU

# 7D12 VHH 序列
sequence = "QVQLVESGGGLVQVGGSLRLSRALSGFWYNHMGWFRQAPGKEREGVAVITADSGSTTYADSVKGRFTISRDDARNTVYLQMNSLKPEDTAVYYCAAGGVGWPYFDYWGQGTQVTVSS"

print(f"预测序列: {sequence}")
print(f"长度: {len(sequence)} aa")

In [ ]:
# 运行 ESMFold 预测（通常 1-2 分钟）
with torch.no_grad():
    output = model.infer_pdb(sequence)

# 保存 PDB 文件
pdb_filename = "7d12_esmfold.pdb"
with open(pdb_filename, "w") as f:
    f.write(output)

print(f"\n预测完成！PDB 文件已保存: {pdb_filename}")

In [ ]:
# 可视化 ESMFold 结果
view = py3Dmol.view(width=800, height=600)
view.addModel(output, 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})

# 高亮 CDR 区域
# CDR1: 25-31 (GFWYNH)
# CDR2: 48-56 (ITADSGST)
# CDR3: 94-106 (AAGGVGWPYFDY)
view.addStyle({'resi': '25-31'}, {'stick': {'colorscheme': 'greenCarbon'}})
view.addStyle({'resi': '48-56'}, {'stick': {'colorscheme': 'orangeCarbon'}})
view.addStyle({'resi': '94-106'}, {'stick': {'colorscheme': 'redCarbon'}})

view.zoomTo()
view.show()

print("\nCDR 区域颜色：")
print("绿色: CDR1 (25-31)")
print("橙色: CDR2 (48-56)")
print("红色: CDR3 (94-106)")

## 方法 3: IgFold (抗体特异性预测) - 推荐用于抗体 ⭐⭐

In [ ]:
# 安装 IgFold
!pip install -q igfold
!pip install -q py3Dmol

In [ ]:
from igfold import IgFoldRunner
from igfold.refine.pyrosetta_ref import init_pyrosetta
import py3Dmol

# 初始化 IgFold
igfold = IgFoldRunner()

# 7D12 VHH 序列
sequence = "QVQLVESGGGLVQVGGSLRLSRALSGFWYNHMGWFRQAPGKEREGVAVITADSGSTTYADSVKGRFTISRDDARNTVYLQMNSLKPEDTAVYYCAAGGVGWPYFDYWGQGTQVTVSS"

print(f"序列: {sequence}")
print(f"长度: {len(sequence)} aa")

In [ ]:
# 运行 IgFold 预测（VHH 格式）
# 对于 VHH，我们只需要重链序列
pred = igfold.fold(
    "7D12_VHH",
    sequences={"H": sequence},
    do_refine=True,  # 使用 PyRosetta 优化
    do_renum=True    # 使用 IMGT 编号
)

# 保存 PDB 文件
pdb_filename = "7d12_igfold.pdb"
pred.to_pdb(pdb_filename)

print(f"\n预测完成！PDB 文件已保存: {pdb_filename}")
print(f"\n置信度分数 (pLDDT): {pred.plddt.mean():.2f}")

In [ ]:
# 可视化 IgFold 结果
with open(pdb_filename, 'r') as f:
    pdb_str = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_str, 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})

# 高亮 CDR 区域
view.addStyle({'resi': '25-31'}, {'stick': {'colorscheme': 'greenCarbon'}})  # CDR1
view.addStyle({'resi': '48-56'}, {'stick': {'colorscheme': 'orangeCarbon'}})  # CDR2
view.addStyle({'resi': '94-106'}, {'stick': {'colorscheme': 'redCarbon'}})  # CDR3

# 高亮 VHH hallmark 残基
view.addStyle({'resi': '37'}, {'sphere': {'color': 'yellow', 'radius': 0.5}})  # F37
view.addStyle({'resi': '44'}, {'sphere': {'color': 'yellow', 'radius': 0.5}})  # E44
view.addStyle({'resi': '45'}, {'sphere': {'color': 'yellow', 'radius': 0.5}})  # R45
view.addStyle({'resi': '47'}, {'sphere': {'color': 'yellow', 'radius': 0.5}})  # E47

view.zoomTo()
view.show()

print("\n图例：")
print("绿色: CDR1 (25-31)")
print("橙色: CDR2 (48-56)")
print("红色: CDR3 (94-106)")
print("黄色球: VHH Hallmark 残基 (F37, E44, R45, E47)")

## 方法 4: 下载 PDB 文件

如果实验结构可用，可以从 PDB 数据库下载：

In [ ]:
# 7D12 VHH 的 PDB ID: 4KRM (链 B)
import requests

pdb_id = "4KRM"
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

response = requests.get(url)
if response.status_code == 200:
    with open(f"{pdb_id}.pdb", "w") as f:
        f.write(response.text)
    print(f"\n已下载 PDB 文件: {pdb_id}.pdb")
    print("注意：这是复合物结构，需要提取 VHH 链")
else:
    print(f"下载失败: {response.status_code}")

## 比较不同方法的结果

In [ ]:
# 如果使用了多种方法，可以比较结果
import os

methods = []
if os.path.exists("7d12_alphafold_results"):
    methods.append("AlphaFold2 (ColabFold)")
if os.path.exists("7d12_esmfold.pdb"):
    methods.append("ESMFold")
if os.path.exists("7d12_igfold.pdb"):
    methods.append("IgFold")
if os.path.exists("4KRM.pdb"):
    methods.append("实验结构 (PDB: 4KRM)")

print("可用的结构预测方法：")
for i, method in enumerate(methods, 1):
    print(f"{i}. {method}")

print("\n推荐使用顺序：")
print("1. IgFold - 专为抗体设计，速度快，准确性高")
print("2. AlphaFold2 - 最准确，但需要较长时间")
print("3. ESMFold - 快速预测，适合初步分析")

## 分析结构特征

In [ ]:
# 使用 BioPython 分析结构
!pip install -q biopython

from Bio.PDB import PDBParser
from Bio.PDB.DSSP import DSSP
import numpy as np

# 解析 PDB 文件（使用 IgFold 结果作为示例）
if os.path.exists("7d12_igfold.pdb"):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("7D12", "7d12_igfold.pdb")
    
    # 计算二级结构
    model = structure[0]
    dssp = DSSP(model, "7d12_igfold.pdb", dssp="mkdssp")
    
    # 统计二级结构
    ss_types = {}
    for key in dssp.keys():
        ss = dssp[key][2]
        ss_types[ss] = ss_types.get(ss, 0) + 1
    
    print("二级结构组成：")
    for ss, count in sorted(ss_types.items()):
        print(f"  {ss}: {count} 残基")
    
    # 计算 CDR 区域的溶剂可及表面积
    print("\nCDR 区域分析：")
    cdr_regions = {
        "CDR1": (25, 31),
        "CDR2": (48, 56),
        "CDR3": (94, 106)
    }
    
    for cdr_name, (start, end) in cdr_regions.items():
        sasa_sum = 0
        for key in dssp.keys():
            resnum = key[1][1]
            if start <= resnum <= end:
                sasa_sum += dssp[key][3]  # SASA
        print(f"  {cdr_name} ({start}-{end}): 总 SASA = {sasa_sum:.1f} Å²")